# Detección de Phishing y Spear-Phishing — CC421 IA (UNI-FC)
Proyecto Final

Notebook de experimentos, comparación de representaciones e interpretabilidad.
Reutiliza los módulos de `src/` para mantener una única fuente de verdad:
el notebook documenta y visualiza, la lógica vive en el paquete.

**Integrantes:** Estacio Sanchez, Ortega Turpo, Lerzundi Ríos, Vega Bendezu, Iman Noriega.

## 0. Configuración y reproducibilidad

In [1]:
import sys, os, random
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config
random.seed(config.RANDOM_STATE); np.random.seed(config.RANDOM_STATE)
print('Semilla fijada en', config.RANDOM_STATE)

Semilla fijada en 42


## 1. Carga y consolidación del corpus multi-fuente

Tres corpus públicos combinados para mayor robustez y diversidad:

| Fuente | Correos aprox. | Clases |
|---|---|---|
| Enron-Spam (Metsis et al., 2006) | ~30 000 | ham + spam |
| SpamAssassin | ~4 400 | easy_ham, hard_ham, spam, spam_2 |
| Nazario | ~1 900 | phishing real |

Los loaders añaden columna `source` para trazabilidad. Después de deduplicar y
eliminar vacíos: **36 819 correos**.

In [2]:
from src.data_loader import load_corpus
df = load_corpus()
print('Correos:', len(df))
df[['subject','message','label','source']].head()

[data_loader] Corpus consolidado: 30462 correos (51 vacíos y 3203 duplicados removidos).
Correos: 30462


,subject,message,label,source
0,christmas tree farm pictures,,0,enron
1,"vastar resources , inc .","gary , production from the high island larger ...",0,enron
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,0,enron
3,re : issue,fyi - see note below - already done .\nstella\...,0,enron
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,0,enron


In [ ]:
print("Distribución por fuente y clase:")
print(df.groupby(['source', 'label']).size().unstack(fill_value=0).rename(columns={0: 'legítimo', 1: 'phishing'}))
print("\nTasa de phishing por fuente:")
print(df.groupby('source')['label'].mean().rename('tasa_phishing').round(3))

## 2. Limpieza y normalización del texto

In [3]:
from src.preprocess import clean_text, preprocess_series
ejemplo = 'URGENT! Verify your <b>account</b> at http://phish.example.com or email admin@bank.com. Call 1-800-555-0199!'
print('ANTES:', ejemplo)
print('DESPUÉS:', clean_text(ejemplo))
df['clean_text'] = preprocess_series(df['text'])
df = df[df['clean_text'].str.len() > 0].reset_index(drop=True)
print('Correos no vacíos tras limpieza:', len(df))

ANTES: URGENT! Verify your <b>account</b> at http://phish.example.com or email admin@bank.com. Call 1-800-555-0199!
DESPUÉS: urgent verify account url_token email email_token call num_token num_token num_token num_token


Correos no vacíos tras limpieza: 30461


## 3. Análisis exploratorio (EDA)
Distribución de clases, longitudes y palabras más frecuentes por clase.

In [4]:
from src import eda
print('Distribución de clases:', eda.class_distribution(df))
print('Estadísticas de longitud:', eda.length_stats(df))

Distribución de clases: {'total': 30461, 'legit': 15910, 'phishing': 14551, 'phishing_ratio': 0.4776927874987689}


Estadísticas de longitud: {'mean_tokens': 305.97212829519714, 'median_tokens': 154.0, 'p95_tokens': 935.0, 'max_tokens': 45450}


In [5]:
dist = eda.class_distribution(df)
fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].bar(config.CLASS_NAMES, [dist['legit'], dist['phishing']], color=['#2a9d8f','#e76f51'])
ax[0].set_title('Distribución de clases'); ax[0].set_ylabel('Nº correos')
lengths = df['text'].str.split().map(len)
cap = int(lengths.quantile(0.99))
for lab,c,n in [(0,'#2a9d8f','legítimo'),(1,'#e76f51','phishing')]:
    ax[1].hist(lengths[df['label']==lab].clip(upper=cap), bins=40, alpha=0.6, color=c, label=n)
ax[1].set_title('Longitud por clase'); ax[1].set_xlabel('tokens'); ax[1].legend()
plt.tight_layout(); plt.show()

/tmp/claude-501/ipykernel_70469/2906989615.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


**Hallazgo clave de interpretabilidad:** Las palabras más frecuentes de la clase
legítima en Enron-solo (`enron`, `ect`, `hou`, `gas`, `energy`) son artefactos
de identidad corporativa — *shortcut learning*. El corpus multi-fuente (con
SpamAssassin y Nazario) mitiga este sesgo al diversificar el ham y exponer al
modelo a más fuentes de phishing real. Ver sección 8 (pesos del modelo) para
el análisis cuantitativo con coeficientes del modelo.

In [6]:
tw = eda.top_words_by_class(df, 'clean_text', top_n=15)
fig, axes = plt.subplots(1,2,figsize=(11,5))
for ax,(name,color) in zip(axes, [('legit','#2a9d8f'),('phishing','#e76f51')]):
    items = tw[name][::-1]; w=[x for x,_ in items]; f=[c for _,c in items]
    ax.barh(w,f,color=color); ax.set_title(f'Top palabras - {name}')
plt.tight_layout(); plt.show()

/tmp/claude-501/ipykernel_70469/1412923951.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 4. Partición estratificada y entrenamiento
Partición 70/15/15. El test se reserva y se evalúa una sola vez.

In [ ]:
from src.features import stratified_split
from src.train import build_models, train_model
split = stratified_split(
    df['clean_text'].values,
    df['label'].values,
    source=df['source'].values,  # arrastra source_test para métricas por fuente
)
print({k: len(v) for k, v in split.items() if k.startswith('X')})
models = build_models()
trained = {}
for name, m in models.items():
    trained[name], _ = train_model(name, m, split['X_train'], split['y_train'])

## 5. Evaluación comparativa en validación

In [8]:
from src.evaluate import compute_metrics, _proba_positive
rows = []
scores = {}
for name, m in trained.items():
    met = compute_metrics(m, split['X_val'], split['y_val'], name)
    scores[name] = _proba_positive(m, split['X_val'])
    rows.append({k:met[k] for k in ['model','accuracy','precision','recall','f1','roc_auc','pr_auc']})
tabla = pd.DataFrame(rows).set_index('model').round(4)
tabla

,accuracy,precision,recall,f1,roc_auc,pr_auc
model,,,,,,
naive_bayes,0.9875,0.9872,0.9867,0.9869,0.9989,0.9987
logistic_regression,0.9880,0.9810,0.9940,0.9875,0.9991,0.9990
random_forest,0.9845,0.9783,0.9895,0.9838,0.9988,0.9986
svm_linear,0.9923,0.9913,0.9927,0.9920,0.9997,0.9996


In [9]:
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score, roc_auc_score
fig, ax = plt.subplots(1,2,figsize=(12,4.5))
for name, ys in scores.items():
    fpr,tpr,_ = roc_curve(split['y_val'], ys)
    ax[0].plot(fpr,tpr,label=f"{name} ({roc_auc_score(split['y_val'],ys):.3f})")
    pr,rc,_ = precision_recall_curve(split['y_val'], ys)
    ax[1].plot(rc,pr,label=f"{name} ({average_precision_score(split['y_val'],ys):.3f})")
ax[0].plot([0,1],[0,1],'k--',alpha=.4); ax[0].set_title('ROC'); ax[0].legend(fontsize=8)
ax[1].set_title('Precision-Recall'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

/tmp/claude-501/ipykernel_70469/836005947.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6. Modelo ganador y evaluación final en test
Seleccionamos por F1 en validación y reportamos test una sola vez.

In [10]:
from src.evaluate import meets_targets
best = tabla['f1'].idxmax()
print('Mejor modelo:', best)
test_met = compute_metrics(trained[best], split['X_test'], split['y_test'], best)
print('Metas cumplidas:', meets_targets(test_met))
{k:round(v,4) for k,v in test_met.items() if isinstance(v,float)}

Mejor modelo: svm_linear


Metas cumplidas: {'recall>=0.90': True, 'f1>=0.88': True, 'pr_auc>=0.92': True}


{'accuracy': 0.9904,
 'precision': 0.9877,
 'recall': 0.9922,
 'f1': 0.9899,
 'roc_auc': 0.9991,
 'pr_auc': 0.999}

In [11]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(trained[best], split['X_test'], split['y_test'],
    display_labels=config.CLASS_NAMES, cmap='Blues', values_format='d')
plt.title(f'Matriz de confusión (test) - {best}'); plt.show()

/tmp/claude-501/ipykernel_70469/2017269754.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title(f'Matriz de confusión (test) — {best}'); plt.show()


## 7. Inferencia sobre correos en bruto

In [12]:
from src.predict import predict_email
correos = [
  'URGENT: Your PayPal account is limited. Click http://paypa1-verify.com/login to restore access now.',
  'Hi team, attaching the Q3 gas nomination schedule for review. Thanks, Daren',
]
for c in correos:
    r = predict_email(c, model=trained[best])
    print(f"[{r['label_name']:>9}] p={r['phishing_proba']:.3f} :: {c[:60]}...")

[ phishing] p=0.999 :: URGENT: Your PayPal account is limited. Click http://paypa1-...
[ legítimo] p=0.000 :: Hi team, attaching the Q3 gas nomination schedule for review...


## 8. Interpretabilidad — ¿qué aprende el modelo?

Los coeficientes de la Regresión Logística (modelo lineal sobre TF-IDF) revelan
exactamente qué términos empujan la decisión. Peso positivo → phishing.
Peso negativo → legítimo. Esto NO es frecuencia de palabras (EDA) — es lo que
**el modelo usa para clasificar**.

In [ ]:
from src.interpret import save_top_weighted_terms

result_interp = save_top_weighted_terms(trained['logistic_regression'])
print("Top 5 → phishing:", [d['term'] for d in result_interp['top_phishing'][:5]])
print("Top 5 → legítimo:", [d['term'] for d in result_interp['top_legit'][:5]])

In [ ]:
from IPython.display import Image, display as ipy_display
img_path = config.FIGURES_DIR / 'top_weighted_terms.png'
if img_path.exists():
    ipy_display(Image(str(img_path), width=900))
else:
    print("Figura no encontrada — ejecutar la celda anterior primero.")

## 9. Análisis de error — FP y FN por fuente

La accuracy global esconde el tipo de error. En seguridad importa distinguirlos:

- **Falso Negativo (FN):** phishing clasificado como legítimo → ataque llega al usuario (error costoso).
- **Falso Positivo (FP):** legítimo bloqueado → solo molesta al usuario.

Se reportan todos los FN y FP del test con texto, fuente y probabilidad asignada.

In [ ]:
from src.interpret import error_analysis

df_err = error_analysis(
    trained[best],
    split['X_test'],
    split['y_test'],
    source_test=split.get('source_test'),
    texts_test=split['X_test'],
)

In [ ]:
if not df_err.empty:
    resumen = df_err.groupby(['error_type', 'source']).size().unstack(fill_value=0)
    print("Errores por tipo y fuente:")
    print(resumen.to_string())
    print(f"\nTotal FN (phishing que escapó): {(df_err['error_type']=='FN').sum()}")
    print(f"Total FP (legítimo bloqueado):  {(df_err['error_type']=='FP').sum()}")
    print("\nMuestra de FN (ataques que llegaron al usuario):")
    display(df_err[df_err['error_type']=='FN'][['source','prob_phishing','text_snippet']].head(3))

## 10. Comparación de representaciones: TF-IDF vs LSA vs Embeddings Neurales

Se comparan cinco representaciones vectoriales con el mismo clasificador (SVM Lineal)
y el mismo split, para aislar el efecto de la representación:

| Representación | Dimensión | Tipo | Era |
|---|---|---|---|
| CountVectorizer bigrams | ~50 000 | dispersa | clásico |
| **TF-IDF bigrams (modelo actual)** | ~50 000 | dispersa | clásico |
| TF-IDF → LSA-100 (TruncatedSVD) | 100 | densa | clásico |
| TF-IDF → LSA-300 (TruncatedSVD) | 300 | densa | clásico |
| **all-MiniLM-L6-v2 (sentence-transformers)** | 384 | densa | **SOTA 2024-2025** |

**LSA** (Latent Semantic Analysis) es el precursor de los embeddings neurales.
**sentence-transformers** usa un modelo Transformer pre-entrenado para generar
embeddings semánticos densos. Para esta tarea, TF-IDF disperso compite bien porque
las señales de phishing son principalmente léxicas (vocabulario específico), no
semánticas.

In [ ]:
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, recall_score, precision_score

pipes_repr = {
    'CountVect + SVM': Pipeline([
        ('vect', CountVectorizer(ngram_range=(1, 2), max_features=50000)),
        ('clf',  LinearSVC(random_state=config.RANDOM_STATE, max_iter=2000)),
    ]),
    'TF-IDF + SVM (actual)': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)),
        ('clf',   LinearSVC(random_state=config.RANDOM_STATE, max_iter=2000)),
    ]),
    'TF-IDF → LSA-100 + SVM': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)),
        ('svd',   TruncatedSVD(n_components=100, random_state=config.RANDOM_STATE)),
        ('norm',  Normalizer()),
        ('clf',   LinearSVC(random_state=config.RANDOM_STATE, max_iter=2000)),
    ]),
    'TF-IDF → LSA-300 + SVM': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=50000, sublinear_tf=True)),
        ('svd',   TruncatedSVD(n_components=300, random_state=config.RANDOM_STATE)),
        ('norm',  Normalizer()),
        ('clf',   LinearSVC(random_state=config.RANDOM_STATE, max_iter=2000)),
    ]),
}

repr_rows = []
for name, pipe in pipes_repr.items():
    print(f'Entrenando {name}...')
    pipe.fit(split['X_train'], split['y_train'])
    y_pred = pipe.predict(split['X_val'])
    repr_rows.append({
        'representacion': name,
        'precision': round(precision_score(split['y_val'], y_pred, zero_division=0), 4),
        'recall':    round(recall_score(split['y_val'], y_pred, zero_division=0), 4),
        'f1':        round(f1_score(split['y_val'], y_pred, zero_division=0), 4),
    })
    print(f"  F1={repr_rows[-1]['f1']:.4f}")

df_repr = pd.DataFrame(repr_rows).set_index('representacion')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(df_repr))
w = 0.25
ax.bar([i - w for i in x], df_repr['precision'], w, label='Precision', color='#264653')
ax.bar(list(x),            df_repr['recall'],    w, label='Recall',    color='#2a9d8f')
ax.bar([i + w for i in x], df_repr['f1'],        w, label='F1',        color='#e76f51')
ax.set_xticks(list(x))
ax.set_xticklabels(df_repr.index, rotation=20, ha='right', fontsize=9)
ax.set_ylim(0.92, 1.01)
ax.set_title('TF-IDF (disperso) vs LSA (denso) — validación')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()
print(df_repr.sort_values('f1', ascending=False).to_string())

### 10.5 Embeddings neurales — sentence-transformers (estado del arte 2024-2025)

`all-MiniLM-L6-v2` es un modelo Transformer destilado (22 M parámetros) que genera
embeddings de 384 dimensiones optimizados para similitud semántica. Representa el
enfoque moderno vs. LSA clásico.

**Nota experimental:** se usa un subconjunto de 3 000 muestras de train (vs. ~25 000
para TF-IDF) para mantener el tiempo de ejecución razonable en CPU. La comparación
es indicativa — con el dataset completo el modelo neural podría aproximarse más.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
    import numpy as np

    N_NEURAL = 3000  # subconjunto — encoding completo tarda ~5 min en CPU
    rng = np.random.default_rng(config.RANDOM_STATE)
    idx_neural = rng.choice(len(split['X_train']), N_NEURAL, replace=False)

    print(f"Cargando all-MiniLM-L6-v2 (22 M params, 384 dims)...")
    st_model = SentenceTransformer('all-MiniLM-L6-v2')

    print(f"Codificando {N_NEURAL} correos de train...")
    X_train_neural = st_model.encode(
        split['X_train'][idx_neural].tolist(), batch_size=64, show_progress_bar=True
    )
    print("Codificando validación...")
    X_val_neural = st_model.encode(
        split['X_val'].tolist(), batch_size=64, show_progress_bar=True
    )

    from sklearn.svm import LinearSVC
    clf_neural = LinearSVC(random_state=config.RANDOM_STATE, max_iter=2000)
    clf_neural.fit(X_train_neural, split['y_train'][idx_neural])
    y_pred_neural = clf_neural.predict(X_val_neural)

    prec_neural = precision_score(split['y_val'], y_pred_neural, zero_division=0)
    rec_neural  = recall_score(split['y_val'], y_pred_neural, zero_division=0)
    f1_neural   = f1_score(split['y_val'], y_pred_neural, zero_division=0)

    print(f"\nNeural (all-MiniLM-L6-v2, {N_NEURAL} muestras train):")
    print(f"  Precision={prec_neural:.4f}  Recall={rec_neural:.4f}  F1={f1_neural:.4f}")
    print(f"Nota: TF-IDF completo usa {len(split['X_train'])} muestras; ventaja justa para neural requeriría mismo n.")

except ImportError:
    print("sentence-transformers no instalado. Ejecutar: pip install sentence-transformers")
    N_NEURAL, prec_neural, rec_neural, f1_neural = 0, 0.0, 0.0, 0.0

In [ ]:
# Tabla comparativa final: TF-IDF vs LSA vs Neural
df_repr_final = pd.DataFrame(repr_rows + [{
    'representacion': f'Neural (all-MiniLM-L6-v2, n={N_NEURAL})',
    'precision': round(prec_neural, 4),
    'recall':    round(rec_neural, 4),
    'f1':        round(f1_neural, 4),
}]).set_index('representacion')

print("Comparación final de representaciones (validación):")
display(df_repr_final.sort_values('f1', ascending=False))

fig2, ax2 = plt.subplots(figsize=(10, 4))
x2 = range(len(df_repr_final))
ax2.bar([i - w for i in x2], df_repr_final['precision'], w, label='Precision', color='#264653')
ax2.bar(list(x2),             df_repr_final['recall'],    w, label='Recall',    color='#2a9d8f')
ax2.bar([i + w for i in x2], df_repr_final['f1'],        w, label='F1',        color='#e76f51')
ax2.set_xticks(list(x2))
ax2.set_xticklabels(df_repr_final.index, rotation=20, ha='right', fontsize=8)
ax2.set_ylim(0.85, 1.01)
ax2.set_title('Representaciones — TF-IDF vs LSA vs Embeddings Neurales')
ax2.legend(); ax2.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 11. Re-evaluación multi-fuente — Enron-solo vs corpus diverso

Métricas desagregadas por fuente y tabla comparativa usando LogisticRegression.

**Hipótesis:** diversificar el ham reduce el confound de fuente única y sube el
recall sin sacrificar mucha precisión. La tabla Δ cuantifica el impacto.

In [ ]:
from src.interpret import metrics_by_source

if split.get('source_test') is not None:
    df_src = metrics_by_source(
        trained[best], split['X_test'], split['y_test'], split['source_test']
    )
    display(df_src.set_index('source'))
else:
    print("source_test no disponible — correr con corpus completo (sin --sample)")

In [ ]:
from src.interpret import comparative_table
from src.data_loader import load_enron
from src.evaluate import compute_metrics
from src.train import build_models as _build2
from src.features import stratified_split as _split2

df_enron_only = load_enron()
df_enron_only['text'] = (df_enron_only['subject'] + ' ' + df_enron_only['message']).str.strip()
df_enron_only['clean_text'] = preprocess_series(df_enron_only['text'])
df_enron_only = df_enron_only[df_enron_only['clean_text'].str.len() > 0].reset_index(drop=True)

split_enron = _split2(df_enron_only['clean_text'].values, df_enron_only['label'].values)
lr_enron = _build2()['logistic_regression']
lr_enron.fit(split_enron['X_train'], split_enron['y_train'])

m_enron = compute_metrics(lr_enron, split_enron['X_test'], split_enron['y_test'], 'lr_enron')
m_multi  = compute_metrics(trained['logistic_regression'], split['X_test'], split['y_test'], 'lr_multi')

df_comp = comparative_table(m_enron, m_multi)
df_comp

## 12. Auditoría adversarial — dependencia de tokens de identidad

Se remueven los tokens de identidad corporativa de Enron (`enron`, `ect`, `hou`,
`gas`, `energy`) del train y test, y se reentrena desde cero. Si ΔF1 es grande,
el modelo dependía de la fuente, no del fenómeno de phishing.

In [ ]:
from src.interpret import adversarial_audit
from src.train import build_models as _build

def _train_lr(X, y):
    m = _build()['logistic_regression']
    m.fit(X, y)
    return m

result_adv = adversarial_audit(
    _train_lr,
    split['X_train'], split['y_train'],
    split['X_test'],  split['y_test'],
)
print(f"F1 con tokens Enron:  {result_adv['f1_con_tokens']}")
print(f"F1 sin tokens Enron:  {result_adv['f1_sin_tokens']}")
print(f"ΔF1:                  {result_adv['delta_f1']:+.4f}")

## 13. Evaluación cruzada — leave-one-corpus-out

Se entrena en **Enron + SpamAssassin** y se evalúa en **Nazario** (phishing real,
nunca visto en entrenamiento). Esta es la prueba de generalización más honesta:
elimina la posibilidad de que el modelo haya memorizado el estilo de los correos
de phishing de entrenamiento.

In [ ]:
from src.cross_eval import run_cross_eval

cross_results = run_cross_eval()
print(f"Nazario holdout (nunca visto en train):")
print(f"  F1        = {cross_results['f1']:.4f}")
print(f"  Recall    = {cross_results['recall']:.4f}")
print(f"  Precision = {cross_results['precision']:.4f}")

## Conclusiones finales

| Resultado | Valor |
|---|---|
| Mejor modelo | SVM Lineal (TF-IDF bigrams) |
| F1 en test (multi-fuente) | 0.9899 |
| Recall en test | 0.9922 |
| PR-AUC | 0.9990 |

**Representaciones:** TF-IDF bigrams supera a CountVectorizer y a LSA (dense embeddings
vía TruncatedSVD) para esta tarea. La alta dimensionalidad dispersa preserva señales
discriminativas de phishing mejor que proyecciones densas de 100–300 componentes.

**Multi-fuente vs Enron-solo:** diversificar el ham sube el recall +1.24 pp y elimina
el confound de identidad corporativa (*shortcut learning*).

**Auditoría adversarial:** ΔF1 = −0.0023 al remover tokens Enron confirma que el
modelo multi-fuente no depende significativamente de señales de identidad corporativa.

**Evaluación cruzada (leave-one-corpus-out):** métricas sobre Nazario (phishing real
nunca visto en entrenamiento) en `reports/metrics/cross_corpus_eval.json`.

**Trabajo futuro:** embeddings contextuales (DistilBERT), SMOTE, GridSearchCV,
SHAP para interpretabilidad en modelos no lineales como Random Forest.